# Notebook "mainCOG" — Maneuver Detection and Summary (COG/SOG)

This notebook automatically scans a directory of GPS runs, detects sailing maneuvers (tacks, jibes, buoy roundings) based on variations in **Course Over Ground (COG)** and **Speed Over Ground (SOG)**, visualizes trajectories and maneuver segments, and produces a **JSON summary** for each run. Each **run folder** must contain **exactly 1 CSV file** (otherwise the run is skipped) and the CSV filename (without extension) is used as the boat name.

---

## Workflow of this notebook

1. **Iterates over runs** (by date, sailor, run index).  
2. **Loads GPS/telemetry data** from CSV.  
3. **Detects COG change points** using rolling mean comparison.  
4. **Identifies SOG maneuvers** via local minima detection and filters (prominence, duration, TWA thresholds, entry speed, etc.).  
5. **Cross-validates maneuvers**: only SOG minima that coincide with significant COG changes are retained.  
6. **Applies manual skips** (some maneuvers are excluded by hand-picked indices).  
7. **Plots results**: trajectories with COG/SOG highlights, annotated maneuvers.  
8. **Appends results** into a global `summary` list.  
9. **Exports summary** as `summary.json`, containing all valid maneuvers per run.

---

## Output

- **Console logs** with details about analyzed runs and skipped maneuvers.  
- **Visual plots** for trajectories and maneuver segments.  
- **`summary.json` file** with structured results:

```json
[
  {
    "date": "08_06",
    "person": "Gian",
    "run": "08_06_Run1",
    "intervals": [
      {
        "maneuver_index": 1,
        "maneuver_time": 1686223456,
        "maneuver_type": "Tack",
        "duration": 12.5,
        "start_time": 1686223440,
        "end_time": 1686223452
      }
    ]
  }
]

In [8]:
import os
import json
from cog_analysis import analyze_session
from natsort import natsorted

summary = []

# Format: { "person/run": [maneuver_index_to_skip, ...] }
'''
manual_skips = {
    "Gian/08_06_2025_Run1": [7,14],
    "Gian/08_06_2025_Run2": [1,8,14],
    "Gian/08_06_2025_Run3": [1,8,14],
    "Gian/08_06_2025_Run4": [5,12],
    "Gian/08_06_2025_Run5": [1,13],
    "Karl/08_06_2025_Run1": [8],
    "Karl/08_06_2025_Run2": [12],
    "Karl/08_06_2025_Run3": [13],
    "Karl/08_06_2025_Run4": [6],
    "Karl/08_06_2025_Run5": [12],
    "Gian/11_06_2025_Run1": [7,14],
    "Gian/11_06_2025_Run2": [7,13],
    "Gian/11_06_2025_Run3": [7,14],
    "Gian/11_06_2025_Run4": [1,7,14],
    "Gian/11_06_2025_Run5": [7,13],
    "Karl/11_06_2025_Run1": [7],
    "Karl/11_06_2025_Run2": [1,8],
    "Karl/11_06_2025_Run3": [7],
    "Karl/11_06_2025_Run4": [5,6,13],
    "Karl/11_06_2025_Run5": [7,14],
    "Karl/11_06_2025_Run6": [7]
    #exluding only hand-picked maneuvers
    #"Gian/08_06_2025_Run2": [1],
    #"Gian/08_06_2025_Run3": [1],
    #"Gian/08_06_2025_Run5": [1],
    #"Gian/11_06_2025_Run4": [1],
    #"Karl/11_06_2025_Run2": [1],
    #"Karl/11_06_2025_Run4": [5]
}
'''

# base_dir = "../Data_Sailnjord/Maneuvers"
# base_dir = "../Data_Sailnjord/Port Camargue June 2025/Maneuvers"
base_dir = "../Data_Sailnjord/Hyères November 2025/Maneuvers"
# Parcours tous les dossiers de date
for date_folder in sorted(os.listdir(base_dir)):
    date_path = os.path.join(base_dir, date_folder)
    for person_folder in sorted(os.listdir(date_path)):
        person_path = os.path.join(date_path, person_folder)
        for run_folder in natsorted(os.listdir(person_path)):
            run_path = os.path.join(person_path, run_folder)
            csv_files = [f for f in os.listdir(run_path) if f.endswith(".csv")]
            csv_path = os.path.join(run_path, csv_files[0])
            print(csv_path)
            print(f"Analyse: {csv_files[0]} dans {run_path}")
            intervals = analyze_session(
                    boat1_path=csv_path, 
                    boat2_path=None, 
                    cog_threshold=30.0, # detect_COG_changes_rolling_mean
                    window=30,  #detect_COG_changes_rolling_mean

                    sog_smoothing_window=75, #detect_maneuvers_from_sog_minima
                    sog_prominence=2, #detect_maneuvers_from_sog_minima
                    sog_min_duration=10.0, #detect_maneuvers_from_sog_minima
                    twa_threshold=90.0, #detect_maneuvers_from_sog_minima
                    cog_inversion_threshold=45.0, #detect_maneuvers_from_sog_minima
                    min_minima_distance=12.0, #detect_maneuvers_from_sog_minima
                    pre_window = 7.0, 
                    post_window = 7.0,
                    min_sog_entry_threshold = 18.0,

                    #Following parameters useless for maneuvers (only used when there is 2 boats to wompare)
                    top_n_intervals=2, # compute_longest_intervals 
                    min_duration_sec=40.0, #compute_longest_intervals
                    sog_derivative_threshold=0.3, #compute_longest_intervals
                    smoothing_window=100  #compute_longest_intervals
                )

            # Build key for skipping
            run_id = f"{person_folder}/{run_folder}"

            # Apply manual skip filter
            if run_id in manual_skips:
                to_skip = manual_skips[run_id]
                original_intervals = intervals.copy()
                skipped = [m for m in original_intervals if m.get("maneuver_index") in to_skip]
                intervals = [m for m in original_intervals if m.get("maneuver_index") not in to_skip]
                if skipped:
                    skipped_str = ", ".join(str(m["maneuver_index"]) for m in skipped)
                    print(f"Skipped {len(skipped)} maneuver(s) in {run_id}: indices: [{skipped_str}]")
                    print(f"Initial number of maneuvers: {len(original_intervals)}")


            print(f"Number of maneuvers selected: {len(intervals)} \n\n\n\n\n\n")
            
            summary.append({
                "date": date_folder,
                "person": person_folder,
                "run": run_folder,
                "intervals": intervals
            })



# Sauvegarde du résumé
summary_file = "summary.json"
with open(summary_file, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Résumé complet sauvegardé ({len(summary)} runs)")


../Data_Sailnjord/Hyères November 2025/Maneuvers\30_11_2025\Gian\30_11_2025_Run11\Gian Stragiotti.csv
Analyse: Gian Stragiotti.csv dans ../Data_Sailnjord/Hyères November 2025/Maneuvers\30_11_2025\Gian\30_11_2025_Run11


KeyError: 'TWA'

In [ ]:
print(json.dumps(summary, indent=2))

[]
